## Integración con Langchain y langgraph

In [141]:
from dotenv import load_dotenv
import os

import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

from openai import OpenAI

import ast

from IPython.display import HTML, Markdown, display
import pandas as pd

# Add this line to load variables from .env file
load_dotenv()

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')  # Retrieve the API key
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')  # Retrieve the API key
LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY')  # Retrieve the API key
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

mod_alto = 'gpt-4.1-2025-04-14' # toma ~30 segs para responder correctamente
mod_bajo = 'gpt-4.1-nano-2025-04-14' # no puede
mod_med = 'gpt-4.1-mini-2025-04-14' # no puede


client = OpenAI()

client.embeddings.create(
  model='text-embedding-3-large', #"text-embedding-ada-002",
  input="The food was delicious and the waiter...",
  encoding_format="float"
)

embedding_fun_openai = OpenAIEmbeddingFunction(api_key=os.environ.get('OPENAI_API_KEY'), model_name="text-embedding-ada-002")

In [142]:
! git branch

* db_f1
  master
  test_enc_1
  test_enc_2


In [143]:
# Install and configure ChromaDB with DuckDB+Parquet backend
from chromadb.config import Settings  
from chromadb import PersistentClient                             # CHANGED
from chromadb.config import Settings, DEFAULT_TENANT, DEFAULT_DATABASE  # CHANGED

# Reinitialize Chroma client with DuckDB+Parquet backend (new ChromaDB API)
from chromadb import Client
from chromadb.config import Settings

DB_PERSIST_DIR = "./db_f1"
DB_NAME        = "enc_dbf1"


client = PersistentClient(
    path     = DB_PERSIST_DIR,
    settings = Settings(allow_reset=True),   # allow resetting on-disk state
    tenant   = DEFAULT_TENANT,
    database = DEFAULT_DATABASE,
) 

print("Collections:", [c.name for c in client.list_collections()])  

db_f1 = client.get_or_create_collection(
    name               = DB_NAME,
    embedding_function = embedding_fun_openai
)

db_f1.peek()

# db_f1 = client.get_collection(name="enc_dbf1")

Collections: ['enc_dbf1']


{'ids': ['p1_1a_1|IDE__question',
  'p1_1a_1|IDE__summary',
  'p1_1a_1|IDE__implications',
  'p2_1a_1|IDE__question',
  'p2_1a_1|IDE__summary',
  'p2_1a_1|IDE__implications',
  'p3_1|IDE__question',
  'p3_1|IDE__summary',
  'p3_1|IDE__implications',
  'p3_2|IDE__question'],
 'embeddings': array([[-0.01286233, -0.02477617,  0.00845953, ..., -0.00216455,
         -0.01462473, -0.00672276],
        [-0.00771893,  0.00366222,  0.03773989, ..., -0.00310992,
         -0.00857366, -0.0436573 ],
        [-0.02147629,  0.00293677,  0.03042583, ...,  0.00184429,
         -0.02441142, -0.04355532],
        ...,
        [ 0.01838034,  0.00962718,  0.02427666, ..., -0.00525712,
         -0.02070235, -0.05458009],
        [-0.00535758,  0.0059622 ,  0.01842068, ..., -0.00686241,
         -0.02355321, -0.02821548],
        [-0.00732108, -0.0118375 ,  0.03220749, ..., -0.00837447,
         -0.01164658, -0.00516163]], shape=(10, 1536)),
 'documents': ['IDENTIDAD_Y_VALORES|Con la palabra maíz, yo asocio

In [144]:
# clientes para encoding de query, etc

client = OpenAI()

client.embeddings.create(
  model='text-embedding-3-large', #"text-embedding-ada-002",
  input="The food was delicious and the waiter...",
  encoding_format="float"
)

embedding_fun_openai = OpenAIEmbeddingFunction(api_key=os.environ.get('OPENAI_API_KEY'), model_name="text-embedding-ada-002")

In [145]:
import pickle

ruta_enc= '/Users/salvadorVMA/Google Drive/01 Proyectos/2025/navegador/encuestas/'
ruta_rep= '/Users/salvadorVMA/Google Drive/01 Proyectos/2025/navegador/reportes/'
ruta_tmp_images= '/Users/salvadorVMA/Google Drive/01 Proyectos/2025/navegador/tmp_images/'

with open(f'{ruta_enc}/los_mex_dict.pkl', 'rb') as f:
    los_mex_dict = pickle.load(f)
    print('los_mex_dict cargado --  leer readme_dict para info')


enc_dict = los_mex_dict['enc_dict']
enc_nom_dict = los_mex_dict['enc_nom_dict']
rev_enc_nom_dict = {v: k for k, v in enc_nom_dict.items()}

pregs_dict = los_mex_dict['pregs_dict']
ses_dict = los_mex_dict['ses_dict']
mkdown_tables = los_mex_dict['mkdown_tables']
df_tables = los_mex_dict['df_tables']

# TODO: usar df_tables para plots

los_mex_dict cargado --  leer readme_dict para info


### query v2: creación de resúmenes temáticos

In [146]:
# query de la base de embeddings
# query_emb es el embedding de la pregunta

user_query= '¿qué significa ser mexicano para los mexicanos?'
# turn it into a vector
query_emb = embedding_fun_openai([user_query])[0]
query_emb

array([-0.01175261, -0.02690859,  0.01918369, ...,  0.00272391,
       -0.02097107, -0.0183757 ], shape=(1536,), dtype=float32)

In [147]:
# ejemplo: query entre las preguntas

result_q = db_f1.query(
    query_embeddings = [query_emb],
    n_results        = 5,
    where            = {"type": "question"}
)

print(result_q["documents"])   # the matched question strings
print(result_q["metadatas"])   # metadata for each hit (includes qid & type)


[['CULTURA_POLITICA|¿Qué tan orgulloso se siente de ser mexicano?', 'IDENTIDAD_Y_VALORES|¿Qué tan orgulloso se siente de ser mexicano?', 'GLOBALIZACION|¿Qué tan orgulloso está de ser mexicano?', 'FEDERALISMO|¿Qué tan orgulloso se siente usted de ser mexicano?', 'INDIGENAS|¿Cuál cree que es la mayor ventaja de ser indígena en México?']]
[[{'qid': 'p5|CUL', 'type': 'question'}, {'qid': 'p6|IDE', 'type': 'question'}, {'qid': 'p15|GLO', 'type': 'question'}, {'qid': 'p19_3|FED', 'type': 'question'}, {'qid': 'p13|IND', 'type': 'question'}]]


In [148]:
# tmp_dist_df contiene las distancias para los tres tipos/ facets, normalizadas entre 0 y 1

# OJO: esto obviamente asume que los tres tipos de información tienen la misma importancia, lo cual no es inicialmente cierto.
# Pero la mezcla de las tres calificaciones devuelve una variedad más amplia de resultados.  

type_lst = [ "question", "summary", "implications"]

tmp_dist_dict = {}
for type in type_lst:
    print(f"Querying for type: {type}")
    tmp_res_q = db_f1.query(
        query_embeddings = [query_emb],
        n_results        = 100,  # devuelvo 100 resultados para cada tipo con distancias menores
        where            = {"type": type}
    )
    [tmp_res_ids] = tmp_res_q['ids']
    [tmp_res_distances] = tmp_res_q['distances']

    tmp_dist_dict[type]= dict(zip(tmp_res_ids, tmp_res_distances))

# remove the suffixes from the keys
tmp_dist_dict = { outer_key: { k.split('__')[0]: v for k, v in inner_dict.items() }
    for outer_key, inner_dict in tmp_dist_dict.items() }

# Create a DataFrame where keys in every subdict are the index and keys in tmp_dist_dict are columns
tmp_dist_df = pd.DataFrame.from_dict(tmp_dist_dict)

# Normalize each column so that max = 1 and min = 0
tmp_dist_df = (tmp_dist_df - tmp_dist_df.min()) / (tmp_dist_df.max() - tmp_dist_df.min())

Querying for type: question
Querying for type: summary
Querying for type: implications


In [149]:
# tmp_res_st contiene las 40 preguntas más cercanas y la descripción de sus resultados.

top_vals = 30

tmp_dist_df['mean'] = tmp_dist_df.mean(axis=1)
tmp_dist_df.sort_values(by='mean', ascending=True, inplace=True)

top_ids = tmp_dist_df.head(top_vals).index.tolist()

tmp_list = []

for type in type_lst:
    for id in top_ids:
        tmp_list.append(id + f'__{type}')


# Retrieve documents using the list of ids
result_by_ids = db_f1.get(ids=tmp_list)

tmp_pre_res_dict = dict(zip(result_by_ids['ids'], result_by_ids['documents']))

tmp_preproc_dic = {k: v for k, v in tmp_pre_res_dict.items() if k.split('__')[1] in ['question', 'summary'] }

# # instead of your current list-comp do this:
tmp_combined_strings = []

 # Calculate the group index based on the position

for i, (k, v) in enumerate(tmp_preproc_dic.items(), start=1):
    facet = k.split("__", 1)[1].upper()
    p_id = k#.split("__")[0].split("|")[0]

    grouped_index = (i + 1) // 2 
    # only split once, and if there is no "|" take the entire v
    q_id = v.split("|")[0]
    parts = v.split("|", 1)
    text = parts[1] if len(parts) > 1 else parts[0]

    p_id = p_id.split("__")[0]

    tmp_combined_strings.append(f"{facet}_{grouped_index}|{p_id}: {text}")

tmp_res_st = '\n'.join(tmp_combined_strings)
tmp_res_st

"QUESTION_1|p1_1a_1|IDE: Con la palabra maíz, yo asocio comida, mercado, animales. Dígame por favor, tres palabras que asocies con la palabra MÉXICO. 1° MENCIÓN\nSUMMARY_1|p1_1a_1|IDE: The table presents various associations that people have with the word 'México', showcasing a wide range of responses. The most prominent association is 'Orgullo', which reflects a strong sense of national pride at 25.25%, followed by 'Costumbres y tradiciones' at 13.50%, and 'Historia y cultura' at 7.58%. Notably, the 'No sabe/ No contesta' response is at 9.75%, indicating a significant portion of respondents either do not have an association or are unwilling to share. This highlights potential gaps in connection or sentiment towards the national identity.\nQUESTION_2|p2_1a_1|IDE: Y ahora, voy a pedir que me diga, por favor, tres palabras que asocie con la palabra MEXICANO. 1° MENCIÓN\nSUMMARY_2|p2_1a_1|IDE: The table presents various words and phrases associated with the identity of 'Mexicano', indicat

In [150]:
tmp_SEL_lst = list(set([st.split('__')[0] for st in tmp_preproc_dic.keys()]))
tmp_RES_lst= []
for ky_st in tmp_SEL_lst:
    tmp_var_st = ky_st
    tmp_dict = {k:v for k,v in tmp_preproc_dic.items() if k.startswith(tmp_var_st)}
    tmp_RES_st = '\n'.join([f' * QUESTION_ID: {tmp_var_st}', 
                            f' QUESTION: {tmp_dict[tmp_var_st+"__question"].split("|")[1]}',
                            f' SUMMARY: {tmp_dict[tmp_var_st+"__summary"]}', 
                        ])
    tmp_RES_lst.append(tmp_RES_st)

tmp_PRMT_st= '\n'.join(tmp_RES_lst)
print(tmp_PRMT_st)

 * QUESTION_ID: p16_1|GLO
 QUESTION: ¿Qué tan orgulloso está usted de México en cada uno de los siguientes ámbitos? El funcionamiento de la democracia
 SUMMARY: The table presents the levels of pride Mexicans feel towards their country's democracy. A majority (35.25%) feel 'Poco orgulloso' (a little proud), followed by those who are 'Algo orgulloso' (29.50%). Only 7.17% report being 'Muy orgulloso' (very proud), while a significant 24.75% express that they are 'Nada orgulloso' (not proud at all). Additionally, the 'No sabe/No contesta' (does not know/no answer) responses, at 1.67%, do not exceed 2.0%, indicating a relatively low level of indecision among respondents regarding their feelings towards democracy in Mexico.
 * QUESTION_ID: p50_6|MIG
 QUESTION: En comparación con los mexicanos, en general, para un extranjero que vive en México ¿es más fácil, igual o más difícil realizar trámites?
 SUMMARY: La tabla muestra las percepciones de los extranjeros que viven en México sobre la faci

### investigación de patrones lógicos (tst_lgc_dict)

In [ ]:
# # propmt optiizado para usar el parser de pydantic

# def create_prompt_crosssum(user_query, tmp_res_st, n_topics=5, format_instructions=""):
#     """
#     Creates a prompt for analyzing a table and answering a query.

#     Parameters:
#         user_query (str): The query to be answered.
#         tmp_res_st (str): The results to analyze.
#         n_topics (int): The number of similar and different patterns to identify.
#         format_instructions (str): The format instructions generated by the PydanticOutputParser.

#     Returns:
#         str: The generated prompt.
#     """

#     prompt = f"""
#     You are a very thorough research assistant that is working on a survey research project.
#     The objective of this project is to study public opinion on a variety of topics.
#     You are fully bilingual in English and Spanish, and can do its work in either language. But you will reply in English.

#     Your task is to read the survey RESULTS and find the top {n_topics} SIMILAR PATTERNS across the RESULTS that best answer the QUERY below.
#     Note SIMILAR PATTERNS are those that show agreement across different questions and summaries, and that are relevant to the QUERY. 
#     You will also return {n_topics} DIFFERENT PATTERNS that are relevant to the QUERY, but that show a pattern that contradicts any of the SIMILAR PATTERNS.

#     IMPORTANT: the results will have the following format: each record will start with '*', and will contain three fields: 
#     QUIESTON_ID, QUESTION, and SUMMARY.
#     The QUESTION_ID is the unique identifier of the question, and it will be used to refer to the results in the PATTERNS. 
#     QUESTION_IDS of the format 'question_id|table_id' (ie, pxx|ABC where xx is any combination of numbers and '_' and YY is any combination of letters, eg. 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG' are valid combiantions)
#     QUESTION_IDS are used to identify the questions and their results. They are not relevant to your answer, but you will need to keep track of these QUESTION_IDS of format pxx|YYY in your RETURN OBJECT.

#     For example, in the following results:

#     TEST_QUERY: 'Do people like sweets?'

#     TEST_RESULTS:
#     * QUESTION_ID: p1_1|ABC: 
#     QUESTION: 'Do you like ice cream?'
#     SUMMARY: 85% of people like ice cream, while 10% said they do not like it, and 5% said they do not know.

#     * QUESTION_ID: p2_1|ABC:
#     QUESTION: 'Do you like marshmallow treats?'
#     SUMMARY: 80% of people like marshmallow treats, while 15% said they do not like it, and 5% said they do not know.
#     * QUESTION_ID: p10_1|DEF:
#     QUESTION: 'Do you like sour apple candies?'
#     SUMMARY: 40% of people like sour apple candies, while 50% said they do not like it, and 10% said they do not know.

#     In these cases, the SIMILAR PATTERN is that 'people really like sweet treats because 85% like ice cream and 80% like marshmallow treats', 
#     and the DIFFERENT PATTERN is that 'people do not like sour treats as much because only 40% said they liked them'.

#     You will describe the PATTERNS in a simple, but detailed way. 
#     Be sure to include as many relevant facts (numbers) as possible and describe the pattern and the supporting facts. 
#     Only include facts you are sure about, and for every fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).
    
#     You will write a VARIABLE_STRING with that will include all QUESTION_IDS you used in the PATTERN, but separating using ',' eg 'p1_1|ABC,p2_1|ABC,p10_1|DEF'.
#     Only include the QUESTION_IDs of the variables you used to create the PATTERN and that you mention in it.

#     Note that any mention of results not available or 'NaN' is NOT THE SAME as 'Do not know' or 'no answer'. 
#     You should ignore any reference to results that are not available or 'NaN'.
#     If you discuss two similar categories together (e.g., 'like a lot' and 'like somehow'), only mention the sum of these percentages if you have all the original ones to sum them. Otherwise, do not mention the figure itself. 
#     Do not discuss figures or percentages you are not sure about.

#     Be careful to not be repetitive and to add a varied list of PATTERNS: if you have discussed any SIMILAR PATTERN or DIFFERENT PATTERN, do not add it to the list again, but come up with a different one. 
    
#     QUERY: {user_query}
#     RESULTS: {tmp_PRMT_st}

#     {format_instructions}
#     """
#     return prompt

# # Create the prompt
# prompt = create_prompt_crosssum(user_query="What does it mean to be Mexican?", tmp_res_st=tmp_res_st, n_topics=5, format_instructions=pattern_format_instructions)

# print(prompt)

In [152]:
## generación del esquema de pydantic para la respuesta

from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser

# 1. Define Pydantic model for an entry
class PatternSummaryEntry(BaseModel):
    SIMILAR_PATTERN_1: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    SIMILAR_PATTERN_2: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    SIMILAR_PATTERN_3: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    SIMILAR_PATTERN_4: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    SIMILAR_PATTERN_5: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    DIFFERENT_PATTERN_1: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    DIFFERENT_PATTERN_2: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    DIFFERENT_PATTERN_3: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    DIFFERENT_PATTERN_4: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    DIFFERENT_PATTERN_5: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }

# 2. Create parser for single entries
pattern_parser_summary = PydanticOutputParser(pydantic_object=PatternSummaryEntry)
pattern_format_instructions = pattern_parser_summary.get_format_instructions()

In [153]:
def create_prompt_crosssum(user_query, tmp_res_st, n_topics=5, format_instructions=""):
    """
    Optimized prompt for extracting non-empty, detailed patterns from survey results.
    """
    prompt = f"""
You are a research assistant analyzing survey results to answer the QUERY below.

Your job is to:
- Identify the top {n_topics} SIMILAR PATTERNS (trends or agreements) and {n_topics} DIFFERENT PATTERNS (contrasts or contradictions) relevant to the QUERY.
- For each pattern, provide:
    1. TITLE_SUMMARY: A short, descriptive title (never empty).
    2. VARIABLE_STRING: Comma-separated QUESTION_IDs used in the pattern (never empty).
    3. DESCRIPTION: A detailed explanation, citing numbers and QUESTION_IDs in parentheses (never empty).

Instructions:
- Do NOT leave any field empty. If information is limited, generalize or summarize what is available.
- Use only facts you are sure about, and always cite the QUESTION_ID for each fact.
- Do NOT repeat the same pattern in multiple fields.
- Ignore any results marked as 'NaN' or not available.
- If you combine categories (e.g., "like a lot" + "like somewhat"), only mention the sum if all original values are present.
- Do NOT invent data; if a pattern is weak, explain that.
- Each pattern must be unique and relevant to the QUERY.

Example input format:
* QUESTION_ID: p1_1|ABC
  QUESTION: 'Do you like ice cream?'
  SUMMARY: 85% of people like ice cream, 10% do not, 5% do not know.
* QUESTION_ID: p2_1|ABC
  QUESTION: 'Do you like marshmallow treats?'
  SUMMARY: 80% like, 15% do not, 5% do not know.
* QUESTION_ID: p10_1|DEF:
  QUESTION: 'Do you like sour apple candies?'
  SUMMARY: 40% of people like sour apple candies, while 50% said they do not like it, and 10% said they do not know.

In these cases, the SIMILAR PATTERN is that 'people really like sweet treats because 85% like ice cream and 80% like marshmallow treats', 
and the DIFFERENT PATTERN is that 'people do not like sour treats as much because only 40% said they liked them'.

Example output for a SIMILAR PATTERN:
TITLE_SUMMARY: High preference for sweet treats
VARIABLE_STRING: p1_1|ABC,p2_1|ABC
DESCRIPTION: A large majority like ice cream (85%, p1_1|ABC) and marshmallow treats (80%, p2_1|ABC).

Note that the QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where xx is any combination of numbers and '_', and YYY are any three capital letters). 
There are examples of valid QUESTION_IDS: 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG'. Be careful to include all the elements of the QUESTION_ID.


Example output (strict JSON, no markdown, no code block, no extra text):
{{
  "SIMILAR_PATTERN_1": {{"TITLE_SUMMARY": "...", "VARIABLE_STRING": "...", "DESCRIPTION": "..."}},
  "SIMILAR_PATTERN_2": {{"TITLE_SUMMARY": "...", "VARIABLE_STRING": "...", "DESCRIPTION": "..."}},
  ...
  "DIFFERENT_PATTERN_5": {{"TITLE_SUMMARY": "...", "VARIABLE_STRING": "...", "DESCRIPTION": "..."}}
}}

QUERY: {user_query}
RESULTS: {tmp_res_st}

{format_instructions}

Checklist before submitting:
- [ ] All fields are filled for each pattern.
- [ ] Each DESCRIPTION includes at least one number and QUESTION_ID.
- [ ] No field is left empty.
"""
    return prompt
prompt = create_prompt_crosssum(user_query="What does it mean to be Mexican?", tmp_res_st=tmp_res_st, n_topics=5, format_instructions=pattern_format_instructions)

print(prompt)


You are a research assistant analyzing survey results to answer the QUERY below.

Your job is to:
- Identify the top 5 SIMILAR PATTERNS (trends or agreements) and 5 DIFFERENT PATTERNS (contrasts or contradictions) relevant to the QUERY.
- For each pattern, provide:
    1. TITLE_SUMMARY: A short, descriptive title (never empty).
    2. VARIABLE_STRING: Comma-separated QUESTION_IDs used in the pattern (never empty).
    3. DESCRIPTION: A detailed explanation, citing numbers and QUESTION_IDs in parentheses (never empty).

Instructions:
- Do NOT leave any field empty. If information is limited, generalize or summarize what is available.
- Use only facts you are sure about, and always cite the QUESTION_ID for each fact.
- Do NOT repeat the same pattern in multiple fields.
- Ignore any results marked as 'NaN' or not available.
- If you combine categories (e.g., "like a lot" + "like somewhat"), only mention the sum if all original values are present.
- Do NOT invent data; if a pattern is wea

In [154]:
import tiktoken
import re
import json

def get_answer(prompt, system_prompt=None, model='gpt-4o-mini-2024-07-18', temperature=0.9):
    # This function sends the prompt to the LLM and retrieves the response.
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user",   "content": prompt})
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content


def clean_llm_json_output(content):
    # Remove code block markers and any leading/trailing text
    content = content.strip()
    # Remove code block markers
    content = re.sub(r"^```json", "", content, flags=re.IGNORECASE).strip()
    content = re.sub(r"^```", "", content, flags=re.IGNORECASE).strip()
    content = re.sub(r"```$", "", content, flags=re.IGNORECASE).strip()
    # Remove any leading text before the first {
    idx = content.find('{')
    if idx > 0:
        content = content[idx:]
    # Remove trailing text after last }
    idx2 = content.rfind('}')
    if idx2 > 0:
        content = content[:idx2+1]
    return content

def get_structured_summary(model_name: str = mod_alto, temperature: float = 0.9):
    # This function combines the prompt creation and LLM call,
    # then parses the response using the PydanticOutputParser.
    prompt =create_prompt_crosssum(user_query=user_query, tmp_res_st=tmp_res_st, n_topics=5, format_instructions=pattern_format_instructions)
    content = get_answer(prompt, model=model_name, temperature=temperature)
    content = clean_llm_json_output(content)
    parsed = pattern_parser_summary.parse(content)
    cleaned = clean_llm_json_output(content)
    try:
        parsed = pattern_parser_summary.parse(cleaned)
        return cleaned, parsed.model_dump()
    except Exception as e:
        print("Parsing failed. Raw output:")
        print(content)
        print("Cleaned output:")
        print(cleaned)
        print("Error:", e)
        return cleaned, {}

content, tst_lgc_dict = get_structured_summary(model_name = mod_alto, temperature= 0.2)
print(content)

# 


{
  "SIMILAR_PATTERN_1": {
    "TITLE_SUMMARY": "Strong and Widespread National Pride",
    "VARIABLE_STRING": "p6|IDE,p5|CUL,p15|GLO,p19_3|FED",
    "DESCRIPTION": "Across multiple questions, a large majority of respondents express strong pride in being Mexican: 63.42% feel 'mucho orgullo' (p6|IDE), 59.67% feel 'mucho orgullo' (p5|CUL), 89.25% are 'very proud' or 'proud' (p15|GLO), and 63.58% are 'very proud' (p19_3|FED). This indicates that pride is a central element of Mexican identity."
  },
  "SIMILAR_PATTERN_2": {
    "TITLE_SUMMARY": "Family and Traditions as Core Elements of Mexican Identity",
    "VARIABLE_STRING": "p1_1a_1|IDE,p2_1a_1|IDE,p19_1|MIG,p13|IND,p13a_1|IND",
    "DESCRIPTION": "Family and traditions are repeatedly cited as key aspects of being Mexican: 'Orgullo' and 'Costumbres y tradiciones' are top associations with 'México' (25.25% and 13.50%, p1_1a_1|IDE), 'Familia' is the most missed aspect abroad (68.47%, p19_1|MIG), and traditions are seen as the main advant

In [155]:
tst_lgc_dict

{'SIMILAR_PATTERN_1': {'TITLE_SUMMARY': 'Strong and Widespread National Pride',
  'VARIABLE_STRING': 'p6|IDE,p5|CUL,p15|GLO,p19_3|FED',
  'DESCRIPTION': "Across multiple questions, a large majority of respondents express strong pride in being Mexican: 63.42% feel 'mucho orgullo' (p6|IDE), 59.67% feel 'mucho orgullo' (p5|CUL), 89.25% are 'very proud' or 'proud' (p15|GLO), and 63.58% are 'very proud' (p19_3|FED). This indicates that pride is a central element of Mexican identity."},
 'SIMILAR_PATTERN_2': {'TITLE_SUMMARY': 'Family and Traditions as Core Elements of Mexican Identity',
  'VARIABLE_STRING': 'p1_1a_1|IDE,p2_1a_1|IDE,p19_1|MIG,p13|IND,p13a_1|IND',
  'DESCRIPTION': "Family and traditions are repeatedly cited as key aspects of being Mexican: 'Orgullo' and 'Costumbres y tradiciones' are top associations with 'México' (25.25% and 13.50%, p1_1a_1|IDE), 'Familia' is the most missed aspect abroad (68.47%, p19_1|MIG), and traditions are seen as the main advantage of being indigenous (

### generación de resúmenes expertos (structured_expert_results)

In [156]:
from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser

# Define the Pydantic model for the output
class ExpertSummaryResponse(BaseModel):
    EXPERT_REPLY: str

# Create the parser
expert_summary_parser = PydanticOutputParser(pydantic_object=ExpertSummaryResponse)

# Generate format instructions
expert_summary_format_instructions = expert_summary_parser.get_format_instructions()

In [157]:
def create_prompt_expt_smry(tst_lgc_dict, tmp_ky, format_instructions=""):
    """
    Creates a prompt for analyzing expert statements and survey summaries.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        tmp_ky (str): Key for the current logical group.
        format_instructions (str): The format instructions generated by the PydanticOutputParser.

    Returns:
        str: The generated prompt.
    """

    ky = tmp_ky

    # Variables identified by the model
    tst_str_lst = tst_lgc_dict[ky]['VARIABLE_STRING'].split(',')
    tst_str_lst = [st + '__question' for st in tst_str_lst]
    print(f'Variables identified by the model: {tst_str_lst}')

    # Variables in the database
    tmp_db_var_lst = db_f1.get(ids=tst_str_lst)['ids']
    tmp_db_var_lst = list(set(tmp_db_var_lst))
    print(f'Variables in the database: {tmp_db_var_lst}')

    # Hallucinated variables
    tmp_hlc_var_lst = set(tst_str_lst) - set(tmp_db_var_lst)
    if tmp_hlc_var_lst:
        print(f'Hallucinated variables by the model: {tmp_hlc_var_lst}')

    # Implications for the identified variables
    tmp_imp_lst = [st.replace('question', 'implications') for st in tmp_db_var_lst]
    tmp_ky_lst = [st.split('__')[0] for st in tmp_imp_lst]
    tmp_imp_dict = dict(
        zip(
            tmp_ky_lst,
            db_f1.get(ids=tmp_imp_lst)['documents']
        )
    )

    imp_st = ' * '.join(tmp_imp_dict.values())
    tmp_smry = tst_lgc_dict[ky]['DESCRIPTION']

    # Generate the prompt
    prompt = f"""
    You are a very thorough research assistant that is working on a survey research project.
    The objective of this project is to study public opinion on a variety of topics.
    You are fully bilingual in English and Spanish, and can do your work in either language. But you will reply in English.
    
    Your task is to read the EXPERT STATEMENTS from one or more experts about what information they consider important to receive from the survey results. 
    Then you will read a SURVEY SUMMARY of some survey results and will write a one-paragraph reply to the experts, elaborating on how the results relate and illustrate their concerns. 
    Note that they are experts and expect a detailed and thorough reply. Reply to all of them in the same paragraph, but be sure to address all of their points. 
    Do not refer to the experts or to yourself in the first person, just write the paragraph as if it was a report.

    Note that the SURVEY SUMMARY will have QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where x, which may include strings like x_x where x is a number, and YYY is a table id). 
    They are not relevant to your answer, but you will need to keep track of these QUESTION_IDS of format pxx|YYY in your RETURN OBJECT.
    There are examples of valid QUESTION_IDS: 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG'. Be careful to include all the elements of the QUESTION_ID.


    Be sure to include as many relevant facts (numbers) as possible and for each fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).

    EXPERT STATEMENTS: * {imp_st}
    SURVEY SUMMARY: {tmp_smry}

    {format_instructions}
    """
    return prompt

tmp_ky = 'SIMILAR_PATTERN_1'

# Generate the prompt
prompt = create_prompt_expt_smry(
    tst_lgc_dict=tst_lgc_dict,
    tmp_ky=tmp_ky,
    format_instructions=expert_summary_format_instructions
)

print(prompt)

Variables identified by the model: ['p6|IDE__question', 'p5|CUL__question', 'p15|GLO__question', 'p19_3|FED__question']
Variables in the database: ['p5|CUL__question', 'p6|IDE__question', 'p15|GLO__question', 'p19_3|FED__question']

    You are a very thorough research assistant that is working on a survey research project.
    The objective of this project is to study public opinion on a variety of topics.
    You are fully bilingual in English and Spanish, and can do your work in either language. But you will reply in English.

    Your task is to read the EXPERT STATEMENTS from one or more experts about what information they consider important to receive from the survey results. 
    Then you will read a SURVEY SUMMARY of some survey results and will write a one-paragraph reply to the experts, elaborating on how the results relate and illustrate their concerns. 
    Note that they are experts and expect a detailed and thorough reply. Reply to all of them in the same paragraph, but b

In [158]:
import tqdm
import tiktoken

def get_structured_expert_summary(tst_lgc_dict, tmp_ky, model_name="gpt-4o-mini-2024-07-18", temperature=0.9):
    """
    Generates a structured expert summary for a given key in the logical group dictionary.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        tmp_ky (str): Key for the current logical group.
        model_name (str): Name of the LLM model to use.
        temperature (float): Temperature setting for the LLM.

    Returns:
        dict: Parsed response as a dictionary.
    """
    # Generate the prompt
    prompt = create_prompt_expt_smry(
        tst_lgc_dict=tst_lgc_dict,
        tmp_ky=tmp_ky,
        format_instructions=expert_summary_format_instructions
    )
    
    # Get the response from the model
    response = get_answer(prompt=prompt, model=model_name, temperature=temperature)
    
    # Parse the response
    parsed = expert_summary_parser.parse(response)
    
    # Ensure the parsed response is returned as a dictionary
    return parsed.model_dump()


def num_tokens_from_string(string: str, encoding_name: str) -> int:
    # Utility function to calculate the number of tokens in a string.
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# Update the token count calculation in batch_documents to use num_tokens_from_string
# 8192 is the token limit for the model
def batch_documents(docs, ids, max_tokens=8192, encoding_name="cl100k_base"):
    # This function batches documents and their IDs while respecting the token limit.
    # It ensures that each batch does not exceed the maximum token limit,
    # which is crucial for efficient processing with the LLM.
    batches = []
    current_batch_docs = []
    current_batch_ids = []
    current_token_count = 0

    for doc, doc_id in zip(docs, ids):
        doc_token_count = num_tokens_from_string(doc, encoding_name)  # Use the token counting function
        
        # print(f"Document ID: {doc_id}, Token Count: {doc_token_count}")
        
        if current_token_count + doc_token_count > max_tokens:
            # Save the current batch and start a new one
            batches.append((current_batch_docs, current_batch_ids))
            current_batch_docs = []
            current_batch_ids = []
            current_token_count = 0

        # Add the document to the current batch
        current_batch_docs.append(doc)
        current_batch_ids.append(doc_id)
        current_token_count += doc_token_count

    # Add the last batch if it has any documents
    if current_batch_docs:
        batches.append((current_batch_docs, current_batch_ids))

    return batches

def batch_process_expert_summaries(tst_lgc_dict, batch_size=8192):
    """
    Processes expert summaries in batches, saving checkpoints after each batch.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        batch_size (int): Maximum token limit for each batch.

    Returns:
        dict: A dictionary of structured results.
    """
    # Prepare prompts and keys
    prompts = [
        create_prompt_expt_smry(tst_lgc_dict, key, format_instructions=expert_summary_format_instructions)
        for key in tst_lgc_dict.keys()
    ]
    keys = list(tst_lgc_dict.keys())
    
    # Batch the documents
    batches = batch_documents(prompts, keys, max_tokens=batch_size, encoding_name="cl100k_base")
    
    # Initialize results dictionary
    structured_results = {}
    
    # Process each batch
    with tqdm.tqdm(total=len(keys), desc="Processing Expert Summaries") as pbar:
        for batch_docs, batch_keys in batches:
            for prompt, key in zip(batch_docs, batch_keys):
                try:
                    # Call get_structured_expert_summary for each key
                    structured_results[key] = get_structured_expert_summary(
                        tst_lgc_dict=tst_lgc_dict,
                        tmp_ky=key,
                        model_name="gpt-4o-mini-2024-07-18",
                        temperature=0.9
                    )
                except Exception as e:
                    structured_results[key] = {'error': str(e)}
                pbar.update(1)

    
    return structured_results

In [159]:

# Process expert summaries in batches
structured_expert_results = batch_process_expert_summaries(tst_lgc_dict)


Variables identified by the model: ['p6|IDE__question', 'p5|CUL__question', 'p15|GLO__question', 'p19_3|FED__question']
Variables in the database: ['p5|CUL__question', 'p6|IDE__question', 'p15|GLO__question', 'p19_3|FED__question']
Variables identified by the model: ['p1_1a_1|IDE__question', 'p2_1a_1|IDE__question', 'p19_1|MIG__question', 'p13|IND__question', 'p13a_1|IND__question']
Variables in the database: ['p13a_1|IND__question', 'p2_1a_1|IDE__question', 'p1_1a_1|IDE__question', 'p19_1|MIG__question', 'p13|IND__question']
Variables identified by the model: ['p2_1a_1|IDE__question']
Variables in the database: ['p2_1a_1|IDE__question']
Variables identified by the model: ['p14a_1|IND__question', 'p32|POB__question', 'p51|MIG__question']
Variables in the database: ['p14a_1|IND__question', 'p32|POB__question', 'p51|MIG__question']
Variables identified by the model: ['p7|IDE__question', 'p53|DEP__question']
Variables in the database: ['p53|DEP__question', 'p7|IDE__question']
Variables id

Processing Expert Summaries:   0%|          | 0/10 [00:00<?, ?it/s]

Variables identified by the model: ['p6|IDE__question', 'p5|CUL__question', 'p15|GLO__question', 'p19_3|FED__question']
Variables in the database: ['p5|CUL__question', 'p6|IDE__question', 'p15|GLO__question', 'p19_3|FED__question']


Processing Expert Summaries:  10%|█         | 1/10 [00:04<00:41,  4.62s/it]

Variables identified by the model: ['p1_1a_1|IDE__question', 'p2_1a_1|IDE__question', 'p19_1|MIG__question', 'p13|IND__question', 'p13a_1|IND__question']
Variables in the database: ['p13a_1|IND__question', 'p2_1a_1|IDE__question', 'p1_1a_1|IDE__question', 'p19_1|MIG__question', 'p13|IND__question']


...

Processing Expert Summaries:  80%|████████  | 8/10 [00:26<00:05,  2.80s/it]

Variables identified by the model: ['p48|MIG__question', 'p51|MIG__question']
Variables in the database: ['p48|MIG__question', 'p51|MIG__question']


Processing Expert Summaries:  90%|█████████ | 9/10 [00:29<00:02,  2.80s/it]

Variables identified by the model: ['p5_1|IDE__question']
Variables in the database: ['p5_1|IDE__question']


Processing Expert Summaries: 100%|██████████| 10/10 [00:34<00:00,  3.40s/it]


### Generación de resúmenes transversales y respuesta a query (final_smry_dict)

In [160]:
from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser

# Define the Pydantic model for the output
class TransversalAnalysisResponse(BaseModel):
    TOPIC_SUMMARIES: dict[str, str]  # A dictionary with topic names as keys and summaries as values
    TOPIC_SUMMARY: str  # A one-paragraph summary for a general audience
    QUERY_ANSWER: str  # A two-sentence answer to the query

# Create the parser
transversal_parser = PydanticOutputParser(pydantic_object=TransversalAnalysisResponse)

# Generate format instructions
transversal_format_instructions = transversal_parser.get_format_instructions()

def create_prompt_trnsvl(tmp_smry_st, user_query, n_cmn_tpc, format_instructions=""):
    """
    Creates a prompt for analyzing expert statements and answering a query.

    Parameters:
        tmp_smry_st (str): The list of expert statements.
        user_query (str): The query to be answered.
        format_instructions (str): The format instructions generated by the PydanticOutputParser.

    Returns:
        str: The generated prompt.
    """
    prompt = f"""
    You are a very thorough research assistant and expert in survey research and public opinion.
    You are fully bilingual in English and Spanish, and can do your work in either language.

    Your task is to perform three analyses and return a single Python dictionary with the results.

    1) Read the following list of STATEMENTS made by experts in several topics, which mention results, their implications, and their relevance.
    Each statement starts with the marker `*` and contains all information for a single statement: results, implications, and relevance.
    Identify {n_cmn_tpc} common topics across the STATEMENTS and write a one-paragraph summary of each topic, citing the most relevant numbers and explanations for each.
    Format your answer as a Python dictionary with the names of the topics as keys in ALL CAPS in the same language as the query, and the summaries as values.

    Note that the STATEMENTS will have QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where x, which may include strings like x_x where x is a number, and YYY is a table id). 
    There are examples of valid QUESTION_IDS: 'p2|ABC', 'p1_1|ABC', 'p20_2|EFG', 'p23_12|EFG'. Be careful to include all the elements of the QUESTION_ID.
    They are not relevant to your answer and your will use them only to identify the variables in the statements that your will use to write the summaries.

    Be sure to include as many relevant facts (numbers) as possible and for each fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).

    Here are the statements: {tmp_smry_st}

    Save your summaries in a Python dictionary with the key `TOPIC_SUMMARIES`.

    2) Read your `TOPIC_SUMMARIES` and write a one-paragraph summary of the most relevant results and implications of the survey results, written for a general audience.
    Be sure to include as many relevant facts (numbers) as possible and for each fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).
    Save your summary in a Python dictionary with the key `TOPIC_SUMMARY`.

    3) Read the QUERY and your `TOPIC_SUMMARY` and write a two-sentence answer to the QUERY that summarizes the most important points of your `TOPIC_SUMMARY`.
    Do not include any numbers or facts in your answer, just your reply to the QUERY.
    Be concise and do not repeat numbers; just answer the QUERY with the relevant ideas.
    
    Here is the QUERY: {user_query}

    Save your answer in a Python dictionary with the key `QUERY_ANSWER`.

    IMPORTANT: Your replies for all three tasks must be in the language of the QUERY.
    IMPORTANT: Make sure to return only a correctly formatted Python dictionary, without any code block formatting, markdown, or additional text.

    {format_instructions}
    """
    return prompt


#

tmp_smry_st = ' * '.join([v['EXPERT_REPLY'] for v in structured_expert_results.values()])
tmp_smry_st

def get_transversal_analysis(tmp_smry_st, user_query, model_name='gpt-4o-mini-2024-07-18', temperature=0.9):
    # Generate the prompt
    prompt = create_prompt_trnsvl(
        tmp_smry_st=tmp_smry_st,
        user_query=user_query,
        n_cmn_tpc=3,
        format_instructions=transversal_format_instructions
    )
    
    # Get the response from the model
    response = get_answer(prompt=prompt, model=model_name, temperature=temperature)
    parsed = transversal_parser.parse(response)
    # Parse the response
    return parsed.model_dump()  # Returns the parsed response as a dictionary.

# Example usage
final_smry_dict = get_transversal_analysis(tmp_smry_st, user_query)
final_smry_dict


{'TOPIC_SUMMARIES': {'IDENTIDAD NACIONAL': 'Los resultados de la encuesta revelan un fuerte sentido de identidad nacional entre los mexicanos, con un 42.00% identificándose únicamente como mexicanos (p7|IDE) y un 96.92% reportando no tener doble nacionalidad (p53|DEP). Este sentido de identidad es vital para la cohesión social y puede influir en iniciativas culturales y educativas que reflejen la identidad predominante.',
  'ORGULLO NACIONAL': "El orgullo nacional es un tema dominante, con más del 59% de los encuestados expresando 'mucho orgullo' (p6|IDE, p5|CUL, p15|GLO). Sin embargo, la baja satisfacción con instituciones nacionales específicas, donde solo un 7.17% se siente 'muy orgulloso' de la democracia (p16_1|GLO), indica una desconexión que podría ser abordada a través de programas que fomenten la confianza y unidad nacional.",
  'PERCEPCIÓN DE DESIGUALDAD': 'Los resultados también reflejan una percepción significativa de desigualdad y discriminación en México, con un 45.12% id

### Construcción de documento

In [161]:
import re

# extracción de plots

tmp_txt_lst = [final_smry_dict['TOPIC_SUMMARY']] + [st for st in final_smry_dict['TOPIC_SUMMARIES'].values()]

# Regular expression pattern to match question ID and table ID format
pattern = r'\(([pP]\d+(?:_\d+)?(?:\|[A-Z0-9]+))\)'

# Extract matches from each text in tmp_txt_lst
tmp_matches = []
for text in tmp_txt_lst:
    matches = re.findall(pattern, text)
    if matches:
        tmp_matches.extend(matches)

# Remove duplicates while preserving order
tmp_id_lst = list(dict.fromkeys(tmp_matches))
# variables en los resultados que sí existen entre las que fueron pasadas al modelo:

print(f"Found {len(tmp_id_lst)} unique question IDs: {tmp_id_lst}")
# Check for differences between the two sets
print(f'variables identificadas pero no pasadas al modelo: {set(tmp_id_lst) - set(tmp_SEL_lst)}')
print(f'variables pasadas al modelo pero no identificadas: {set(tmp_SEL_lst) - set(tmp_id_lst)}')


Found 3 unique question IDs: ['p7|IDE', 'p53|DEP', 'p16_1|GLO']
variables identificadas pero no pasadas al modelo: set()
variables pasadas al modelo pero no identificadas: {'p13a_1|IND', 'p39|MIG', 'p50_6|MIG', 'p1_1a_1|IDE', 'p43_3|MIG', 'p14a_1|IND', 'p16_3|GLO', 'p33_4|IND', 'p19_3|FED', 'p50_7|MIG', 'p6|IDE', 'p15|GLO', 'p43_4|MIG', 'p53|MIG', 'p51|MIG', 'p19_1|MIG', 'p16_2|GLO', 'p5_1|IDE', 'p65|MIG', 'p50_5|MIG', 'p5|CUL', 'p48|MIG', 'p32|POB', 'p67_1|MIG', 'p13|IND', 'p78|IDE', 'p2_1a_1|IDE'}


In [162]:
# tmp_sel_plt_dict contiene los plots seleccionados


tmp_sel_plt_dict = {st: {'df': df_tables[st]} for st in tmp_id_lst if st in df_tables}
if set(tmp_id_lst) - set(tmp_sel_plt_dict.keys()):
    missing_ids = list(set(tmp_id_lst) - set(tmp_sel_plt_dict.keys()))
    print(f"Warning: Some IDs were not found in df_tables: {missing_ids}")

else:
    print(f"All IDs were found in df_tables.")

All IDs were found in df_tables.


In [ ]:
# creación de gráficos

import numpy as np
import math
import matplotlib.pyplot as plt

def split_text_by_words(text, n=8, k=50):
    """
    Splits a string into lines with a maximum of n words per line.
    If the first n-1 words already exceed k characters, truncates with '...'.
    """
    words = text.split()
    # if first (n-1) words already too long → truncate
    first_chunk = words[: n-1]
    if len(" ".join(first_chunk)) >= k:
        return " ".join(first_chunk) + "..."
    # otherwise chunk every n words
    lines = []
    for i in range(0, len(words), n):
        chunk = words[i : i + n]
        lines.append(" ".join(chunk))
    return "\n".join(lines)

def create_plot(df, question, figsize=(6, 8)):
    """
    Creates a bar plot (horizontal if rows <= 6, vertical otherwise) of the survey data.

    Args:
        df (pd.DataFrame): DataFrame containing the data to plot.
        question (str): The survey question to use as the plot title.

    Returns:
        matplotlib.pyplot.figure: The matplotlib figure object.
    """
    # Determine plot type based on the number of rows
    if len(df) >= 6:
        # Horizontal bar plot
        fig, ax = plt.subplots(figsize=figsize)
        
        # Create a colormap where 'No sabe/ No contesta' is always gray

        colormaps = [plt.cm.summer, plt.cm.spring, plt.cm.winter, plt.cm.autumn]
        selected_colormap = np.random.choice(colormaps)
        colors = selected_colormap(np.linspace(0, 0.8, len(df)))

        if 'No sabe/ No contesta' in df.index:
            gray_index = list(df.index).index('No sabe/ No contesta')
            colors[gray_index] = [0.7, 0.7, 0.7, 1.0]  # RGBA for gray
        bars = ax.barh(df.index, df['%'], color=colors)

        # Add percentage labels to the bars
        for bar in bars:
            width = bar.get_width()
            label_position = width + 0.5  # Slightly to the right of the bar
            ax.text(label_position, bar.get_y() + bar.get_height()/2,
                    f'{width:.1f}%',
                    ha='left', va='center', fontsize=8)
        # Customize plot
        ax.set_xlabel('Porcentaje (%)', fontsize=8)
        ax.set_title(split_text_by_words(question), fontsize=9)
        # Apply split_text_by_words to y-tick labels with conditional fontsize
        y_labels = [split_text_by_words(label, n=8, k=50) for label in df.index]
        y_fontsizes = [10 if len(label.split()) <= 4 else 8 for label in df.index]
        ax.set_yticklabels(y_labels)
        for label, fontsize in zip(ax.get_yticklabels(), y_fontsizes):
            label.set_fontsize(fontsize)

        # Apply split_text_by_words to x-tick labels with conditional fontsize
        x_labels = [split_text_by_words(str(label), n=8, k=50) for label in ax.get_xticks()]
        x_fontsizes = [10 if len(str(label).split()) <= 4 else 8 for label in ax.get_xticks()]
        ax.set_xticklabels(x_labels)
        for label, fontsize in zip(ax.get_xticklabels(), x_fontsizes):
            label.set_fontsize(fontsize)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        # Invert the y-axis to show the first item at the top
        ax.invert_yaxis()
        plt.tight_layout()

    else:
        # Vertical bar plot
        fig, ax = plt.subplots(figsize=figsize)

        colormaps = [plt.cm.summer, plt.cm.spring, plt.cm.winter, plt.cm.autumn]
        selected_colormap = np.random.choice(colormaps)
        colors = selected_colormap(np.linspace(0, 0.8, len(df)))

        if 'No sabe/ No contesta' in df.index:
            gray_index = list(df.index).index('No sabe/ No contesta')
            colors[gray_index] = [0.5, 0.5, 0.5, 1.0]  # RGBA for gray

        bars = ax.bar(df.index, df['%'], color=colors)

        # Add percentage labels above the bars
        for bar in bars:
            height = bar.get_height()
            label_position = height + 0.5
            ax.text(bar.get_x() + bar.get_width()/2, label_position,
                    f'{height:.1f}%',
                    ha='center', va='bottom', fontsize=8)

        # Customize plot
        ax.set_ylabel('Porcentaje (%)', fontsize=8)

        # Apply split_text_by_words to y-tick labels with conditional fontsize
        y_labels = [split_text_by_words(label, n=8, k=50) for label in df.index]
        y_fontsizes = [10 if len(label.split()) <= 4 else 8 for label in df.index]
        ax.set_yticklabels(y_labels)
        for label, fontsize in zip(ax.get_yticklabels(), y_fontsizes):
            label.set_fontsize(fontsize)

        # Apply split_text_by_words to x-tick labels with conditional fontsize
        x_labels = [split_text_by_words(str(label), n=8, k=50) for label in ax.get_xticks()]
        x_fontsizes = [10 if len(str(label).split()) <= 4 else 8 for label in ax.get_xticks()]
        ax.set_xticklabels(x_labels)
        for label, fontsize in zip(ax.get_xticklabels(), x_fontsizes):
            label.set_fontsize(fontsize)
        ax.set_title(split_text_by_words(question, 8), fontsize=9)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()

    return fig

def create_multiplot(dfs, questions=None, plot_idx=1, figsize=(10,5)):
    """
    Creates a grid of subplots (2 cols) for each df in `dfs`.
    Titles each sub-plot "Fig. {plot_idx}" and then increments plot_idx.
    Returns (fig, new_plot_idx).
    """
    n = len(dfs)
    ncols = 2
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(figsize[0], figsize[1]*nrows))
    axes = axes.flatten()

    cmap_list = [plt.cm.summer, plt.cm.spring, plt.cm.winter, plt.cm.autumn]

    for i, df in enumerate(dfs):
        ax = axes[i]
        questions[i] if questions else df.index.name or f"Plot {i+1}"
        # prepare colors, gray out "No sabe/ No contesta"
        cmap = np.random.choice(cmap_list)
        colors = cmap(np.linspace(0,0.8,len(df)))
        if 'No sabe/ No contesta' in df.index:
            gi = list(df.index).index('No sabe/ No contesta')
            colors[gi] = [0.7,0.7,0.7,1.0]

        # choose horiz vs vert
        if len(df) >= 6:
            bars = ax.barh(df.index, df['%'], color=colors)
            for b in bars:
                w = b.get_width()
                ax.text(w+0.5, b.get_y()+b.get_height()/2, f"{w:.1f}%", 
                        ha="left", va="center", fontsize=8)
            ax.set_xlabel("Porcentaje (%)", fontsize=6)
            ax.invert_yaxis()
        else:
            bars = ax.bar(df.index, df['%'], color=colors)
            for b in bars:
                h = b.get_height()
                ax.text(b.get_x()+b.get_width()/2, h+0.5, f"{h:.1f}%", 
                        ha="center", va="bottom", fontsize=8)
            ax.set_ylabel("Porcentaje (%)", fontsize=6)
            ax.tick_params(axis='x', rotation=45)

        # Y-axis
        for lbl in ax.get_yticklabels():
            txt = lbl.get_text()
            # skip numbers
            if not re.fullmatch(r'\d+(\.\d+)?%?', txt):
                lbl.set_text(split_text_by_words(txt, n=3, k=50))
        # X-axis
        for lbl in ax.get_xticklabels():
            txt = lbl.get_text()
            if not re.fullmatch(r'\d+(\.\d+)?%?', txt):
                lbl.set_text(split_text_by_words(txt, n=3, k=50))
        # redraw so the new text is taken into account
        ax.figure.canvas.draw()


        # set the title with current figure index
        ax.set_title(
            split_text_by_words(f"Fig. {plot_idx}\n{questions[i] if questions else df.index.name}", 8),
            fontsize=8
        )
        plot_idx += 1

    # drop any empty axes
    for extra in axes[n:]:
        fig.delaxes(extra)

    plt.tight_layout()
    return fig, plot_idx

def find_vars(v, pattern=r'([pP]\d+(?:[a-zA-Z0-9_]+)?\|[A-Z0-9]+)'):
    """
    Finds and returns all variable IDs in the string v that match the specified pattern,
    including IDs like p1_1a_1|IDE, p2_1a_1|IDE, p14a_1|IND and simpler forms like p6|IDE.

    Returns:
        list: A list of unique variable IDs found in the string.
    """
    if isinstance(v, str):
        matches = re.findall(pattern, v)
        return list(set(matches)) if matches else None
    return None

def fix_title(plot_vars, plot_idx):
    # Fix the title to be more descriptive
    tmp_title = f"Fig. {plot_idx}\n" + ', '.join([pregs_dict.get(var, 'Unknown Question') for var in plot_vars if var in pregs_dict])
    plot_idx += 1
    return tmp_title


ptrn_dict= {ky : structured_expert_results[ky]['EXPERT_REPLY'] for ky in structured_expert_results.keys()}

texts = {}

# 1) construcción del documento
texts = {
    'PREGUNTA: ': user_query,
    'RESPUESTA: ': final_smry_dict['QUERY_ANSWER'],
    'RESUMEN DEL ANÁLISIS': final_smry_dict['TOPIC_SUMMARY']
}

texts.update(final_smry_dict['TOPIC_SUMMARIES'])

# texts.update({'PATRONES EXPERTOS: ': ''})

# texts.update(ptrn_dict)

# identificación de variables para plots
tmp_plot_dict = {ky: find_vars(v) for ky, v in texts.items()}
tmp_dfs_dict = {ky: [df_tables[st] for st in (tmp_plot_dict[ky] or []) if st in df_tables.keys()] for ky in tmp_plot_dict.keys()}

# prepare output dirs
os.makedirs("tmp_images", exist_ok=True)

md_lines = []

plot_idx = 1

for section, text in texts.items():
    md_lines.append(f"# {section}")
    md_lines.append(text)
    plot_vars = tmp_plot_dict.get(section) or []

    if len(plot_vars) == 1:
        var = plot_vars[0]
        df = tmp_dfs_dict.get(section, [None])[0]
        if df is not None:
            title = pregs_dict.get(var, var).split('|')[-1]
            fig = create_plot(df, f'Fig. {plot_idx}\n{title}', figsize=(5,5))
            img = f"tmp_images/{var.replace('|','_').lower()}.png"
            fig.savefig(img, dpi=150); plt.close(fig)
            md_lines.append(f"![{var}]({img})")
            plot_idx += 1

    elif len(plot_vars) > 1:
        # collect the exact dataframes and labels in order
        dfs = tmp_dfs_dict.get(section, [])
        qs  = [pregs_dict.get(var, var).split('|')[-1] for var in plot_vars]
        if dfs:
            fig, plot_idx = create_multiplot(dfs, questions=qs, figsize=(8,4))
            img = f"tmp_images/{section.replace(' ','_').lower()}_multiplot.png"
            fig.savefig(img, dpi=150)
            plt.close(fig)
            md_lines.append(f"![{section} multiplot]({img})")
            plot_idx += 1


# write out the markdown file
with open(ruta_rep + "report.md", "w") as f:
    f.write("\n\n".join(md_lines))

print("✅ report.md and images/ saved.")


# Convert to PDF using pypandoc
import pypandoc
from datetime import datetime
# Create timestamp for unique filename

write_pdf = True

if write_pdf: 
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_pdf = f"report_{timestamp}.pdf"
    try:
        pypandoc.convert_text(
            md_lines,
            to="pdf",
            format="md",
            outputfile=ruta_rep + output_pdf,
            extra_args=[
                "--pdf-engine=xelatex",
                "--variable=geometry:margin=1in",
                "--variable=mainfont:Helvetica",
                "--variable=fontsize:12pt"
            ]
        )
        print(f"PDF report saved to {output_pdf}")
    except Exception as e:
        print(f"Error converting to PDF: {str(e)}")
        print("You can try converting the markdown file manually with Pandoc or another tool.")


# TODO: arreglar pdf y lo demás....

/var/folders/35/kp3vmnl94d9cp_qddwwb50z00000gn/T/ipykernel_6196/1177386570.py:64: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(y_labels)
/var/folders/35/kp3vmnl94d9cp_qddwwb50z00000gn/T/ipykernel_6196/1177386570.py:71: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(x_labels)


✅ report.md and images/ saved.
Error converting to PDF: 'list' object has no attribute 'encode'
You can try converting the markdown file manually with Pandoc or another tool.
